In [1]:
from collections import defaultdict, deque
from itertools import combinations


class PartitionTransformer:
    def __init__(self, source, target):
        self.source = [frozenset(g) for g in source]
        self.target = [frozenset(g) for g in target]

        self._validate()

    def _validate(self):
        s = set().union(*self.source)
        t = set().union(*self.target)

        if s != t:
            raise ValueError("Source und Target enthalten unterschiedliche Variablen.")

    # ------------------------------------------------------------
    # Überlappungsgraph
    # ------------------------------------------------------------

    def build_overlap_graph(self):
        """
        Bipartiter Graph:
            S_i -- T_j
        falls Schnittmenge != leer
        """

        graph = defaultdict(set)

        for i, s in enumerate(self.source):
            for j, t in enumerate(self.target):
                if s & t:
                    graph(("S", i)).add(("T", j))
                    graph(("T", j)).add(("S", i))

        return graph

    def connected_components(self, graph):
        visited = set()
        comps = []

        for node in graph:
            if node in visited:
                continue

            comp = set()
            q = deque([node])
            visited.add(node)

            while q:
                u = q.popleft()
                comp.add(u)

                for v in graph[u]:
                    if v not in visited:
                        visited.add(v)
                        q.append(v)

            comps.append(comp)

        return comps

    # ------------------------------------------------------------
    # Optimaler Transformationsplan
    # ------------------------------------------------------------

    def optimal_plan(self):
        """
        Liefert Liste von Operationen:
            ("split", original, [teile])
            ("join", [gruppen], neue_gruppe)
        """

        ops = []

        current = list(self.source)

        # --------------------------------------------------------
        # Schritt 1:
        # splitte nur entlang der Ziel-Schnittmengen
        # --------------------------------------------------------

        new_current = []

        for group in current:

            intersections = []

            for target_group in self.target:
                inter = group & target_group
                if inter:
                    intersections.append(inter)

            # nichts zu tun
            if len(intersections) == 1:
                new_current.append(group)
                continue

            # optimaler split
            ops.append(("split", tuple(sorted(group)),
                        [tuple(sorted(x)) for x in intersections]))

            new_current.extend(intersections)

        current = new_current

        # --------------------------------------------------------
        # Schritt 2:
        # joins gemäß target
        # --------------------------------------------------------

        current_sets = set(current)

        for target_group in self.target:

            parts = [g for g in current_sets if g <= target_group]

            if len(parts) == 1 and next(iter(parts)) == target_group:
                continue

            if len(parts) > 1:
                ops.append((
                    "join",
                    [tuple(sorted(p)) for p in parts],
                    tuple(sorted(target_group))
                ))

                for p in parts:
                    current_sets.remove(p)

                current_sets.add(target_group)

        return ops

    # ------------------------------------------------------------
    # Minimale Anzahl Operationen
    # ------------------------------------------------------------

    def minimal_operation_count(self):
        graph = self.build_overlap_graph()
        c = len(self.connected_components(graph))

        m = len(self.source)
        n = len(self.target)

        return (m - c) + (n - c)


# ============================================================
# Beispiel
# ============================================================

source = [
    ("a", "b"),
    ("c",),
    ("d", "e")
]

target = [
    ("a",),
    ("c",),
    ("d", "e", "b")
]

pt = PartitionTransformer(source, target)

print("Minimale Anzahl Operationen:")
print(pt.minimal_operation_count())

print("\nOperationsplan:")

for op in pt.optimal_plan():
    print(op)

Minimale Anzahl Operationen:


TypeError: 'collections.defaultdict' object is not callable